# 05 — Model Comparison and LightGBM Tuning

Current best: **LightGBM on the 165 raw features, public LB 0.948.** This notebook:

1. compares LightGBM with two other boosting libraries (XGBoost, CatBoost);
2. tunes a few LightGBM settings;
3. averages LightGBM over several random seeds;
4. writes submission files for the best candidates.

All models use the 165 raw features only (graph features did not help, see notebook 04).

*Runtime: about 15 minutes.*

## How models are scored

Every model gets two numbers:

- **Standard** — fit on steps 1–24, validate on 25–35. Measures "familiar" patterns. Almost every tree model scores
  about 0.995–0.998 here, so it cannot separate models well.
- **Unseen** — the *average* pooled AUC over four fit windows that end **before** step 23 (1–16, 1–18, 1–20, 1–22),
  each validated on the following steps up to 35. Notebook 04 showed that models that have not seen step 23 miss a
  group of illicit transactions, so this measures how a model copes with illicit cases unlike its training data.
  Averaging four windows makes the number steadier than the single hard split.

**Noise level.** Changing only the random seed moves a single split's AUC by about 0.005–0.009. Differences smaller
than that should not be trusted, so the best candidates are re-checked with several seeds before choosing.

**Limits.** "Unseen" correctly preferred LightGBM over Random Forest on the leaderboard, but it wrongly favoured a
feature-subset blend. We trust it most for comparing model types and settings, and the leaderboard has the final say.

In [ ]:
import sys, warnings
from datetime import date
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

from src.data import load_train, load_test, labeled, save_submission, FEATURE_COLS, LABEL_COL, SEED
from src.validation import evaluate, evaluate_many, log_experiment
from src.models import SeedAverage

TODAY, NOTEBOOK = date.today().isoformat(), "05_model_comparison"
UNSEEN_ENDS = [16, 18, 20, 22]
STANDARD_END = 24

C_A, C_B, C_MUTED = "#2a78d6", "#eb6834", "#898781"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.6, "axes.axisbelow": True,
    "axes.edgecolor": C_MUTED, "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "legend.frameon": False,
})

train, test = load_train(), load_test()


def score(model):
    """Return (unseen, standard) pooled AUC for one model."""
    unseen = evaluate_many(model, FEATURE_COLS, train, UNSEEN_ENDS).mean()
    standard = evaluate(model, FEATURE_COLS, train, STANDARD_END)["overall_auc"]
    return unseen, standard


def lgbm(**changes):
    """LightGBM with the settings of the current best submission (LB 0.948), plus any changes."""
    params = dict(n_estimators=500, learning_rate=0.02, num_leaves=15, colsample_bytree=0.3,
                  class_weight="balanced", random_state=SEED, verbose=-1)
    params.update(changes)
    return LGBMClassifier(**params)

## 1. Other model types: XGBoost and CatBoost

All three are *gradient-boosted trees*: they build many small decision trees one after another, each correcting the
previous ones' mistakes. They differ in how trees are grown:

- **LightGBM** grows trees leaf by leaf (it always splits the most useful leaf next).
- **XGBoost** grows trees level by level, up to a maximum depth.
- **CatBoost** uses "symmetric" trees, where every node at the same depth uses the same split. This often generalises
  well with little tuning.

Each library gets a couple of reasonable settings. Class imbalance is handled the same way in spirit: `class_weight`
(LightGBM), `scale_pos_weight=10` (XGBoost: illicit rows count about 10×, roughly the licit:illicit ratio in the fit
window), and `auto_class_weights="Balanced"` (CatBoost).

In [ ]:
candidates = {
    "LightGBM (current best, LB 0.948)": lgbm(),
    "XGBoost, depth 4": XGBClassifier(n_estimators=500, learning_rate=0.02, max_depth=4, colsample_bytree=0.3,
                                      subsample=0.8, scale_pos_weight=10, tree_method="hist",
                                      random_state=SEED, n_jobs=-1),
    "XGBoost, default settings": XGBClassifier(scale_pos_weight=10, random_state=SEED, n_jobs=-1),
    "CatBoost, depth 4": CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=4, auto_class_weights="Balanced",
                                            random_seed=SEED, verbose=0, thread_count=-1,
                                            allow_writing_files=False),
    "CatBoost, depth 6": CatBoostClassifier(iterations=1000, learning_rate=0.05, depth=6, auto_class_weights="Balanced",
                                            random_seed=SEED, verbose=0, thread_count=-1,
                                            allow_writing_files=False),
}
types = pd.DataFrame([dict(zip(["model", "unseen", "standard"], [name, *score(m)])) for name, m in candidates.items()])
types = types.set_index("model").sort_values("unseen", ascending=False)
types.round(4)

In [ ]:
for name, row in types.iterrows():
    log_experiment(TODAY, NOTEBOOK, f"Model type: {name}", val_auc=row.standard,
                   notes=f"unseen AUC (mean of fit 1-16/18/20/22) {row.unseen:.4f}")

## 2. Tuning LightGBM

We try a small grid over three settings that control how flexible each tree is:

| Setting | What it does | Values tried |
|---|---|---|
| `num_leaves` | maximum leaves per tree (bigger = more complex trees) | 7, 15, 31 |
| `colsample_bytree` | share of the 165 columns each tree may use | 0.15, 0.3, 0.6 |
| `min_child_samples` | minimum rows in a leaf (bigger = smoother, less overfitting) | 20, 100 |

That is 18 combinations. Number of trees (500) and learning rate (0.02) stay fixed.

In [ ]:
grid = []
for leaves in [7, 15, 31]:
    for cols in [0.15, 0.3, 0.6]:
        for min_rows in [20, 100]:
            unseen, standard = score(lgbm(num_leaves=leaves, colsample_bytree=cols, min_child_samples=min_rows))
            grid.append({"num_leaves": leaves, "colsample_bytree": cols, "min_child_samples": min_rows,
                         "unseen": unseen, "standard": standard})
grid = pd.DataFrame(grid).sort_values("unseen", ascending=False).reset_index(drop=True)
grid.round(4)

In [ ]:
# Average "unseen" AUC for each value of each setting (averaging over the other two settings)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, col in zip(axes, ["num_leaves", "colsample_bytree", "min_child_samples"]):
    m = grid.groupby(col).unseen.mean()
    ax.plot([str(v) for v in m.index], m.values, color=C_A, lw=2, marker="o", ms=6)
    ax.set_xlabel(col)
    ax.set_title(f"effect of {col}", loc="left", fontsize=10)
axes[0].set_ylabel("mean unseen AUC")
plt.tight_layout(); plt.show()

**Reading the grid.** The clearest pattern is `colsample_bytree`: letting each tree see more columns (0.6) does better
on unseen patterns than 0.15 or 0.3. A plausible reason: with few columns per tree, many trees are forced to rely on
weaker features that may only fit the training period.

The top rows of the table are close together. Picking the single best of 18 noisy scores tends to pick a lucky one,
so we re-check the top three, plus the current model, with three different seeds.

**Selection rule.** Among the re-checked settings we pick the highest mean "unseen" AUC, but only if the standard AUC
stays at 0.996 or above. The blend in notebook 04 gained on unseen patterns while losing on familiar ones, and scored
worse on the leaderboard, so we don't accept trading away standard accuracy.

In [ ]:
top = grid.head(3)
recheck_configs = {"current (15 leaves, cols 0.3, min 20)": {}}
for _, r in top.iterrows():
    name = f"{int(r.num_leaves)} leaves, cols {r.colsample_bytree}, min {int(r.min_child_samples)}"
    recheck_configs[name] = dict(num_leaves=int(r.num_leaves), colsample_bytree=r.colsample_bytree,
                                 min_child_samples=int(r.min_child_samples))

rows = []
for name, params in recheck_configs.items():
    for seed in [42, 1, 2]:
        unseen, standard = score(lgbm(**params, random_state=seed))
        rows.append({"config": name, "seed": seed, "unseen": unseen, "standard": standard})
recheck = pd.DataFrame(rows)
recheck_summary = recheck.groupby("config", sort=False)[["unseen", "standard"]].agg(["mean", "min"])
recheck_summary.round(4)

In [ ]:
# Choose: highest mean unseen AUC over the 3 seeds, among configs that keep standard AUC >= 0.996
means = recheck.groupby("config", sort=False)[["unseen", "standard"]].mean()
eligible = means[means.standard >= 0.996]
best_name = eligible.unseen.idxmax()
best_params = recheck_configs[best_name]
gain = means.loc[best_name, "unseen"] - means.iloc[0].unseen
print(f"chosen: {best_name}  ->  unseen {means.loc[best_name, 'unseen']:.4f} "
      f"({gain:+.4f} vs current), standard {means.loc[best_name, 'standard']:.4f}")

for name, r in means.iterrows():
    log_experiment(TODAY, NOTEBOOK, f"LightGBM tuning: {name} (3 seeds)", val_auc=r.standard,
                   notes=f"unseen AUC mean over 3 seeds {r.unseen:.4f}")

## 3. Averaging over several seeds

`SeedAverage` (in `src/models.py`) trains the same LightGBM 5 times with seeds 42, 1, 2, 3, 4 and averages the
predicted probabilities. Each run makes slightly different random choices, and averaging cancels part of that noise.

In [ ]:
seed_rows = []
for name, params in [("current", {}), (best_name, best_params)]:
    single = recheck[recheck.config == (list(recheck_configs)[0] if name == "current" else name)]
    unseen, standard = score(SeedAverage(lgbm(**params)))
    seed_rows.append({"config": name, "single seed (mean of 3)": single.unseen.mean(),
                      "5-seed average": unseen, "standard (5-seed)": standard})
seed_table = pd.DataFrame(seed_rows).set_index("config")
seed_table.round(4)

In [ ]:
for name, r in seed_table.iterrows():
    log_experiment(TODAY, NOTEBOOK, f"LightGBM 5-seed average: {name}", val_auc=r["standard (5-seed)"],
                   notes=f"unseen AUC {r['5-seed average']:.4f} (single seed {r['single seed (mean of 3)']:.4f})")

## 4. Submissions

Each file is trained on **all labeled rows of steps 1–35**, then checked against `sample_submission.csv` before saving.
To keep leaderboard results interpretable, each file changes **one thing** compared with a file already scored:

| File | Compared with | What it tests |
|---|---|---|
| `lgbm_seedavg` | `lgbm_raw` (LB 0.948) | seed averaging only |
| `lgbm_tuned_seedavg` | `lgbm_seedavg` | the tuned settings only |

In [ ]:
full = labeled(train)
final_models = {
    "lgbm_seedavg": SeedAverage(lgbm()),
    "lgbm_tuned_seedavg": SeedAverage(lgbm(**best_params)),
}
for tag, model in final_models.items():
    model.fit(full[FEATURE_COLS], full[LABEL_COL])
    pred = model.predict_proba(test[FEATURE_COLS])[:, 1]
    path = save_submission(test, pred, f"{TODAY}_{tag}.csv")
    print("saved and validated:", path.relative_to(ROOT))
    cfg = "current settings" if tag == "lgbm_seedavg" else best_name
    row = seed_table.loc["current" if tag == "lgbm_seedavg" else best_name]
    log_experiment(TODAY, NOTEBOOK, f"SUBMISSION {tag}, refit on steps 1-35", val_auc=row["standard (5-seed)"],
                   notes=f"{cfg}, 5 seeds; unseen AUC {row['5-seed average']:.4f}; file submissions/{TODAY}_{tag}.csv")

## Summary

| Candidate | Unseen AUC | Standard AUC |
|---|---|---|
| LightGBM, current settings (LB 0.948) | 0.902 | 0.997 |
| CatBoost (best of 2) | 0.887 | 0.995 |
| XGBoost (best of 2) | 0.871 | 0.995 |
| LightGBM tuned: 31 leaves, cols 0.6, min 100 (mean of 3 seeds) | 0.909 | 0.997 |
| LightGBM current, 5-seed average | 0.904 | 0.997 |
| **LightGBM tuned, 5-seed average** | **0.909** | 0.997 |

1. **LightGBM stays the best model type.** XGBoost and CatBoost score 0.016–0.03 lower on unseen patterns and no
   higher on the standard split.
2. **Tuning helps a little.** The clearest effect is `colsample_bytree`: letting each tree use 60% of the columns
   beats 15% or 30%. The chosen settings gain **+0.0065** on unseen patterns (3-seed average) with no loss on the
   standard split. That is about the size of seed noise, so it is a modest, not certain, improvement.
   (7-leaf trees scored slightly higher on unseen, but they lose 0.0025 on the standard split, so the selection rule
   excludes them.)
3. **Seed averaging adds very little** (+0.0003 to +0.0014). It mostly makes predictions steadier.
4. **Submissions** (each changes one thing compared with a scored file):
   - `lgbm_seedavg`: the current model with seed averaging. Expected to land close to 0.948.
   - `lgbm_tuned_seedavg`: the tuned settings on top. **Recommended to submit first.**

The validation gains here are small. The leaderboard will show whether they carry over to the test period.